In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
BASE_DIR = "/content/drive/MyDrive/FALL 25/CECS 456/Final_project/Content"
ANNOTATIONS_PATH = os.path.join(BASE_DIR, "annotations.xml")
IMAGES_DIR = os.path.join(BASE_DIR, "images")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "crops")



In [7]:
import xml.etree.ElementTree as ET
from PIL import Image

classes = [
    "free_parking_space",
    "not_free_parking_space",
    "partially_free_parking_space",
]

for cls in classes:
    os.makedirs(os.path.join(OUTPUT_DIR, cls), exist_ok=True)

tree = ET.parse(ANNOTATIONS_PATH)
root = tree.getroot()
num_crops = 0

for image_node in root.findall("image"):
    img_name = image_node.get("name")          # e.g. "images/0.png"
    img_path = os.path.join(BASE_DIR, img_name)

    if not os.path.isfile(img_path):
        print("Missing:", img_path)
        continue

    img = Image.open(img_path).convert("RGB")

    for i, poly in enumerate(image_node.findall("polygon")):
        label = poly.get("label")
        if label not in classes:
            continue

        # Parse polygon -> bounding box
        pts = [p.split(",") for p in poly.get("points").split(";")]
        pts = [(float(x), float(y)) for x, y in pts]
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]

        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        crop = img.crop((xmin, ymin, xmax, ymax))

        base_name = os.path.splitext(os.path.basename(img_name))[0]
        out_name = f"{base_name}_{i}.jpg"
        out_path = os.path.join(OUTPUT_DIR, label, out_name)
        crop.save(out_path)
        num_crops += 1

print("DONE. Total crops:", num_crops)

DONE. Total crops: 903
